# Train/Test Split & Cross-Validation

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 3/7

A model graded on questions it already saw will always look brilliant — this lesson
builds the evaluation machinery that keeps our scores honest.

## 🎯 Learning Objectives

- Explain why held-out data must stay untouched until final scoring.
- Control `train_test_split` via `test_size`, `random_state`, and `stratify`.
- Prove class balance survives a stratified split — and breaks without one.
- Execute a full train/validation/test three-way split.
- Run k-fold cross-validation and read fold scores as mean ± std.
- Compare several models fairly inside one cross-validation loop.
- Choose between `KFold`, shuffled `KFold`, and `StratifiedKFold` deliberately.

## 1. Why Held-Out Data Is Sacred

Imagine two students preparing for the HSC exam. One studies **past papers** and
sits a **brand-new** paper. The other memorises the past papers' answer keys and
sits *the same past paper again*. Who learned anything?

A model scored on its own training rows is always the second student. The fix:

| Dataset | Role | Analogy |
|---|---|---|
| **Train** | The model learns patterns here | Past papers you study |
| **Test** | Scored ONCE at the end | The real unseen exam |
| (Validation / CV) | Model selection & tuning | Mock tests before the exam |

The iron rule: **fit on train, score on test** — and never let tuning decisions
peek at the test set, or it silently becomes training material.

**Syntax:**
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # fraction (or count) reserved for testing
    random_state=42,     # reproducible shuffle
    stratify=y,          # keep class proportions identical in both parts
)
```

## 2. `test_size` and `random_state`

`test_size=0.2` holds out every fifth row. `random_state` seeds the shuffle: same
seed, same split, same numbers forever — which is what makes your notebook
reproducible for teammates (and graders).

In [ ]:
# Reproducibility proof: same seed -> identical split; different seed -> different split
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

tr_a, _, y_a, _ = train_test_split(X, y, test_size=0.2, random_state=42)
tr_b, _, y_b, _ = train_test_split(X, y, test_size=0.2, random_state=42)
tr_c, _, y_c, _ = train_test_split(X, y, test_size=0.2, random_state=7)

print("seed 42 twice identical rows? ", tr_a.equals(tr_b))
print("seed 42 vs seed 7 differ?      ", not tr_a.equals(tr_c))
print("\nSplit sizes:", len(tr_a), "/", len(X), "->",
      f"{len(tr_a)/len(X):.0%} train, {1 - len(tr_a)/len(X):.0%} test")

Common sizes: 80/20 when you have thousands of rows; 70/30 or even 60/40 when
data is scarce (the test set must stay big enough to be *believable*).

## 3. `stratify=y`: Keeping Class Proportions

With imbalanced classes, a plain random split can hand the test set a very different
class mix than reality — your score then measures a population that doesn't exist.
`stratify=y` forces both halves to mirror the full dataset's proportions.

```text
Full data:  90% ham / 10% spam
stratify=y  ->  train 90/10 AND test 90/10   ✅
no stratify ->  test might land 96/4         ❌ different world
```

In [ ]:
# Craft an imbalanced target: 90% class 0, 10% class 1
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 500
X_imb = pd.DataFrame({"score_a": rng.normal(50, 10, n),
                      "score_b": rng.normal(30, 5, n)})
y_imb = np.concatenate([np.zeros(450, dtype=int), np.ones(50, dtype=int)])
rng.shuffle(y_imb)

print("FULL data positive rate :", f"{y_imb.mean():.1%}")

_, te_plain, _, y_te_plain = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42)              # NOT stratified
_, te_strat, _, y_te_strat = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)

print("TEST positive rate, plain split    :", f"{y_te_plain.mean():.1%}")
print("TEST positive rate, stratified split:", f"{y_te_strat.mean():.1%}")

The stratified split reproduces 10.0% exactly; the plain split drifts. For rare
disease screening (say 2% positive) a drift like that changes which errors dominate —
and therefore what your accuracy even means.

## 4. The Iron Rule in Action: Fit on Train, Score on Test

One clean pass: learn on train rows only, then measure once on test rows. Watch the
gap between the two scores — that gap *is* the generalisation story from lesson 1.

In [ ]:
# Fit on train, score honestly on test
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

print(f"train accuracy: {tree.score(X_train, y_train):.3f}   <- open-book exam")
print(f"test  accuracy: {tree.score(X_test,  y_test):.3f}   <- closed-book exam")
print(f"gap           : {tree.score(X_train, y_train) - tree.score(X_test, y_test):.3f}")
print("(a deep, unlimited tree memorises some train noise - hence the drop)")

## 5. Three-Way Split: Train / Validation / Test

Two decisions compete for the same data: *fitting* and *tuning*. Give them separate
rooms. Tune hyperparameters against **validation**, report the final number from
**test**, touched exactly once.

| Split | Share (typical) | Used for |
|---|---|---|
| Train | 60% | Fitting parameters |
| Validation | 20% | Choosing hyperparameters / model family |
| Test | 20% | Final, single, honest measurement |

**Syntax:**
```python
# split once into train+tmp vs test, then carve validation out of train+tmp
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.2, ...)
X_train, X_val, y_train, y_val = train_test_split(X_tr, y_tr, test_size=0.25, ...)
# 0.8 * 0.25 = 0.2  ->  60 / 20 / 20 overall
```

In [ ]:
# The 60/20/20 carve-up
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

# Step 1: reserve the test set (never touched while experimenting)
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
# Step 2: carve validation out of what remains
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)

for name, part in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    print(f"{name:<11}{len(part):>4} rows ({len(part)/len(X):.0%})")

When data is scarce (hundreds of rows, not millions), sacrificing 20% permanently
stings. That's the cue to switch to **cross-validation**, which reuses every row for
both training and validation — next section.

## 6. k-Fold Cross-Validation

Split the training pool into `k` equal folds. Train on `k−1` folds, validate on the
remaining one; repeat `k` times so every fold takes one turn as the validator.
Average the `k` scores → far more stable than a single lucky/unlucky split.

```text
fold:    1    2    3    4    5
run 1  [VAL][TR ][TR ][TR ][TR ]
run 2  [TR ][VAL][TR ][TR ][TR ]
...                     ...
run 5  [TR ][TR ][TR ][TR ][VAL]
```

**Syntax:**
```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)   # returns 5 fold accuracies
scores.mean(), scores.std()                    # report as mean ± std
```

> 🔍 **Under the Hood:** `cross_val_score` calls `clone(model)` for every fold —
> each run gets a brand-new, freshly initialised estimator and fits it from zero.
> Your original object is never touched (still unfitted!), nothing leaks between
> runs, and the `k` results are statistically dependent only through sharing data,
> not through shared weights. That clone-per-fold design is why CV measures a
> *procedure* (model + hyperparameters + preprocessing), not one lucky fitted object.

In [ ]:
# 5-fold cross-validation on the full training pool
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

logreg = LogisticRegression(max_iter=10000)
scores = cross_val_score(logreg, X, y, cv=5)

print("fold accuracies:", scores.round(3))
print(f"summary: {scores.mean():.3f} +/- {scores.std():.3f}")
print("(std is the wobble across folds - small std = stable model)")

## 7. Fair Fight: Comparing Models with the Same CV

Every candidate gets identical folds, identical data, identical metric — the only
difference left is the model itself. That is a fair comparison.

In [ ]:
# Three contenders, one referee (5-fold CV each)
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

contenders = {
    "LogisticRegression": LogisticRegression(max_iter=10000),
    "KNN (k=5)":          KNeighborsClassifier(n_neighbors=5),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in contenders.items():
    s = cross_val_score(model, X, y, cv=5)
    rows.append({"model": name, "mean_accuracy": round(s.mean(), 3), "std": round(s.std(), 3)})

board = pd.DataFrame(rows).sort_values("mean_accuracy", ascending=False)
print(board.to_string(index=False))
print("\nNote how unscaled KNN trails - distance models suffer from raw units (lesson 02).")

Reading the board: prefer the top mean, but distrust a large std — a model that
swings wildly between folds may just be lucky on this dataset. And remember KNN's
lag: wrap it with a scaler (lesson 7's Pipelines) and watch it catch up.

## 8. When Shuffled and Stratified KFold Matter

Plain `KFold` slices rows **in order**. If classes happen to be grouped (very common:
files sorted by label, hospital batches), folds become single-class islands and CV
collapses. Two upgrades:

| Splitter | Behaviour | Reach for it when |
|---|---|---|
| `KFold(n_splits=k)` | Contiguous slices, no shuffle | Time-ordered data (avoid shuffling away the timeline!) |
| `KFold(n_splits=k, shuffle=True)` | Random slices, mixed classes by luck | Default-ish for iid data |
| `StratifiedKFold(n_splits=k)` | Each fold mirrors global class ratios | Classification, especially imbalanced |

`cross_val_score(cv=5)` auto-selects StratifiedKFold for classifiers — but know what
you're getting.

In [ ]:
# Sorted labels + plain KFold = disaster; watch each fold's positive rate
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

y_sorted = np.concatenate([np.zeros(45, dtype=int), np.ones(45, dtype=int)])  # 50/50, SORTED
X_dummy = np.zeros((len(y_sorted), 1))

splitters = {
    "plain KFold (no shuffle)":            KFold(n_splits=5),
    "shuffled KFold":                      KFold(n_splits=5, shuffle=True, random_state=42),
    "StratifiedKFold":                     StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
}
for name, splitter in splitters.items():
    rates = [y_sorted[test_idx].mean() for _, test_idx in splitter.split(X_dummy, y_sorted)]
    print(f"{name:<26}", [f"{r:.2f}" for r in rates])

Plain KFold produces folds that are **all-zeros or all-ones** — any model would score
0% or 100% per fold, pure fiction. Shuffling mixes classes; stratifying pins every
fold to the true 50/50. With imbalanced real data (fraud: 0.5% positives), stratified
splitting isn't a nicety, it's the difference between measuring something and
measuring noise.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Scoring on training data | Measures memorisation; always flattering | Hold out test data before fitting |
| Tuning hyperparameters against the test set | Test set degrades into extra training data | Tune on validation / CV; touch test once |
| Omitting `random_state` | Teammates can't reproduce your split (or your numbers) | Always seed the shuffle |
| Skipping `stratify=y` on imbalanced classes | Test set misrepresents the true class mix | Pass `stratify=y` for classification |
| Scaling/imputing before splitting | Transformer statistics leak test info (lesson 02) | Pipeline transformers with the model |
| Plain KFold on sorted/grouped data | Folds become single-class (or single-patient) islands | `shuffle=True` or `StratifiedKFold`; group-aware splitters if rows share sources |

## 💡 Best Practices & Pro Tips

- **Report `mean ± std`, not one fold's number.** A single split can flatter or bury
  a model; five-fold averages tame that lottery.
- **Freeze the test set early** in a project, write its filename down, and don't
  reload it until the final report.
- **Match the splitter to the structure:** time series → `TimeSeriesSplit`;
  multiple rows per patient/user → `GroupKFold`; otherwise stratified.
- **Small data → bigger k** (leave-one-out for tiny datasets) to waste less training signal;
  huge data → a single 80/20 split often suffices because noise averages out anyway.
- **AI-engineering relevance:** offline CV numbers are the gate that decides whether
  a model ships to production — sloppy splitting there means silent regressions live.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `train_test_split(X, y, test_size=0.2)` | Carves out a hold-out set | Add `random_state=42, stratify=y` |
| `stratify=y` | Mirrors class ratios in every split | Essential for classification |
| `cross_val_score(model, X, y, cv=5)` | Scores k folds, returns array | `scores.mean(), scores.std()` |
| `KFold(shuffle=True)` | Mixed-class random folds | Regression / iid data |
| `StratifiedKFold(n_splits=5)` | Class-balanced folds | Imbalanced classification |
| `clone(model)` per fold (automatic) | Fresh estimator each run | Why CV measures procedures |

Key takeaways:
- Training score answers *"did it learn?"*; test/CV score answers *"did it
  generalise?"* — only the second pays rent.
- Seed everything (`random_state`) and stratify classification splits.
- Three-way split when data is rich; k-fold CV when it isn't.
- Same folds for all contenders, or the comparison is theatre.

## 🔗 Next Lesson

Continue to **[04_Linear_Regression](../04_Linear_Regression/notes.ipynb)** —
with honest evaluation in place, meet your first model family: the line.